# AI Image FFT Detector - Colab runner

Run setup first, then only the authorized phase. Phase 1 collects original files; no model training yet. CPU only. Keep printed outputs for team review. All code is published on main.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Clone the project
Clone main from the configured repository. An existing checkout is not overwritten; use a fresh session when you need a new checkout.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Rahul2-5/SpectraCheck.git"
PROJECT_DIR = Path("/content/ai-image-fft-detector")
if REPO_URL == "REPLACE_WITH_YOUR_GITHUB_REPOSITORY_URL":
    raise ValueError("Set REPO_URL to your GitHub repository URL first")
if PROJECT_DIR.exists():
    raise FileExistsError("Checkout already exists. Inspect it or use a fresh Colab session.")
subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
%pip install -r /content/ai-image-fft-detector/requirements.txt
%pip install --no-deps -e /content/ai-image-fft-detector


If Colab asks you to restart after installation, use Runtime > Restart session, then continue below (do not rerun the clone cell). This avoids mixing preloaded library versions with newly installed ones.

In [ ]:
from pathlib import Path
import importlib
import importlib.metadata as metadata
import os
import platform
import subprocess
import sys

PROJECT_DIR = Path("/content/ai-image-fft-detector")
os.chdir(PROJECT_DIR)
print("Python:", platform.python_version())
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
modules = {"pillow": "PIL", "scikit-learn": "sklearn", "pyyaml": "yaml", "imagehash": "imagehash", "huggingface-hub": "huggingface_hub"}
for line in (PROJECT_DIR / "requirements.txt").read_text().splitlines():
    if not line.strip() or line.startswith("#"):
        continue
    package, expected = line.split("==")
    installed = metadata.version(package)
    assert installed == expected, f"Version mismatch: {package}: {installed} != {expected}"
    importlib.import_module(modules.get(package, package))
    print(f"{package}=={installed}")
from aidetect.io_utils import config_hash, load_config
cfg = load_config("config.yaml")
print("Config SHA-256:", config_hash(cfg))


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)


## Phase 0 review
Record setup results and check GitHub Actions. If Phase 1 has been authorized, continue below; otherwise stop here.

## Phase 1 - Original image collection
Download CIFAKE manually from [Kaggle](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images) and extract it on Drive. Set the folder below to the parent of train/REAL, train/FAKE, test/REAL, test/FAKE. The pilot is separate from main data. Keep this session on CPU.

In [ ]:
CIFAKE_SOURCE_DIR = Path("/content/drive/MyDrive/cifake")
if not CIFAKE_SOURCE_DIR.is_dir():
    raise FileNotFoundError("Set CIFAKE_SOURCE_DIR to your extracted CIFAKE folder on Drive")
subprocess.run([sys.executable, "scripts/download_pilot.py", "--source-dir", str(CIFAKE_SOURCE_DIR)], cwd=PROJECT_DIR, check=True)


### Main collection
The next command scans Tiny-GenImage metadata, counts every generator, then saves only SD15 images and a random sample of real images. It may transfer the full upstream Parquet dataset; a subset does not guarantee a small download. Rerun after a disconnection to reuse unchanged saved files. Exit code 2 means the SD15 count is too small: outputs are saved, but consult the supervisor before Phase 2.

In [ ]:
subprocess.run([sys.executable, "scripts/download_main.py"], cwd=PROJECT_DIR, check=True)


In [ ]:
import json
import pandas as pd
from PIL import Image as PILImage
from IPython.display import display

cfg = load_config(PROJECT_DIR / "config.yaml")
for collection in ("pilot", "main"):
    root = Path(cfg["paths"]["data_root"]) / collection
    manifest = pd.read_csv(root / "manifest.csv")
    record = json.loads((root / "acquisition.json").read_text())
    assert record["status"] == "complete", "Acquisition did not finish"
    listed = set(manifest["rel_path"])
    actual = {p.relative_to(root).as_posix() for folder in ("real", "ai") for p in (root / folder).glob("*") if p.is_file() and p.suffix != ".tmp"}
    assert len(manifest) == len(listed) and listed == actual, "Manifest/file counts differ"
    assert all((root / row.rel_path).stat().st_size == row.n_bytes for row in manifest.itertuples())
    print(collection, "saved counts:", manifest.groupby("label").size().to_dict())
    print(collection, "source generator counts:", record["generator_counts"])
    if collection == "main":
        assert record["saved_counts"]["ai"] >= cfg["dataset"]["minimum_ai_images"], "STOP: ask supervisor about insufficient SD15 data"
        print("Record SD15 and SD14 counts in the team tracker.")
    for label in (0, 1):
        samples = manifest[manifest["label"] == label].sample(n=5, random_state=cfg["seed"])
        for relative in samples["rel_path"]:
            with PILImage.open(root / relative) as image:
                image.load()
                print(collection, "label", label, relative, image.size, image.format)
                display(image.copy())


## STOP - Phase 1 team review
Confirm generator counts, supervisor decision if SD15 is below the configured minimum, matching manifest/file counts, and five readable random images per class. Share errors without starting Phase 2. When checks pass, tell Codex to proceed to Phase 2.

## Phase 2 - Audit, duplicates, frozen splits
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 3 - Shared preprocessing
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 4 - Frequency features
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 5 - Pixel baseline and feature extraction
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 6 - Training and tuning
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 7 - Freeze and single test evaluation
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 8 - Local Streamlit prototype
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.

## Phase 9 - Report pack and reproducibility
TODO: add this phase's commands only when authorized. Read AGENTS.md and the PRD first.